## 00. Cấu hình chỉnh sửa tập trung

Đổi các đường dẫn Kaggle Input tại đây. `SMOKE_ONLY=True` chỉ chạy kiểm tra 4 mẫu mỗi split và ba bước cập nhật gate. Bật `SMOKE_ONLY=False` cùng các cờ RUN_* để chạy toàn bộ. Mọi stage có kiểm tra kết quả trước khi tiếp tục.


In [ ]:
from pathlib import Path
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
REPO_REF = "main"  # Hoặc SHA chứa đúng notebook và scripts của MOE-09.5
REPOSITORY_SOURCE = None  # Kaggle Input có source MOESCORE, nếu chưa push mã mới
PROJECT_ROOT = Path("/kaggle/working/MOESCORE")
DATASET_ROOT = Path("/kaggle/input/YOUR_XACLE_DATASET")
WEIGHTS_ROOT = Path("/kaggle/input/YOUR_WEIGHTS_DATASET")
HF_CACHE_SOURCE = Path(
    "/kaggle/input/datasets/qnv8071/"
    "moescore-hf-text-cache/hf_hub_cache"
)

HF_HOME = Path(
    "/kaggle/working/moescore/huggingface"
)
UNILM_SOURCE_DIR = None  # None: dùng bản UniLM đã kiểm chứng trong repo
EXISTING_FEATURE_CACHE = None  # Kaggle Input chứa feature_cache.tar + upload_manifest.json, hoặc feature_cache/
WORK_ROOT = Path("/kaggle/working/moescore")
SMOKE_CACHE_ROOT = WORK_ROOT / "smoke"
FEATURE_CACHE_ROOT = WORK_ROOT / "extraction"
RUNS_ROOT = WORK_ROOT / "runs"
CONFIG_ROOT = WORK_ROOT / "configs"
LOG_ROOT = WORK_ROOT / "logs"
HF_HOME = WORK_ROOT / "huggingface"
UPLOAD_DIRECTORY = WORK_ROOT / "upload"
KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"
FEATURE_DATASET_SLUG = "moescore-full9-feature-cache"
FEATURE_DATASET_HANDLE = f"{KAGGLE_USERNAME}/{FEATURE_DATASET_SLUG}"
EXPERT_PROFILE = "full_9"
GPU_DEVICES = ["cuda:0", "cuda:1"]
WORKER_DEVICES = {"expert1":"cuda:0", "expert2":"cuda:1", "expert3":"cuda:0",
                  "expert4":"cuda:1", "top2_m2d":"cuda:0", "top2_msclap":"cuda:1",
                  "top2_laion":"cuda:0", "top2_mga":"cuda:1", "m2d_lora":"cuda:0"}
CHECKPOINT_OVERRIDES = {}  # Ví dụ: {"m2d_lora.finetuned_checkpoint": "/kaggle/input/.../best.pt"}
EXTRACTION_BATCH_SIZES = {name: 1 for name in WORKER_DEVICES}
SHARD_SIZE = 128
SMOKE_SAMPLES_PER_SPLIT = 4
SMOKE_ONLY = True
RUN_GPU_SMOKE = True
RUN_FULL_EXTRACTION = False
UPLOAD_FEATURE_CACHE = False
RUN_GATE_TRAINING = False
RUN_EVALUATION = False
RUN_TEST_INFERENCE = False
INSTALL_MISSING_DEPENDENCIES = True
WHEELHOUSE = None  # Tùy chọn: thư mục wheel nếu tắt Internet khi cài package
GATE_DEVICE = "cuda:0"
TRAIN_BATCH_SIZE = 8
TRAIN_EPOCHS = 50
TRAIN_LEARNING_RATE = 1e-5
SEED = 2027
assert EXPERT_PROFILE == "full_9" and SMOKE_SAMPLES_PER_SPLIT == 4

## 01. Kiểm tra Python và hai GPU

Chấp nhận Python 3.12 hoặc 3.13 và ít nhất hai CUDA device. Stage này không tự chuyển sang CPU hoặc một GPU.


In [ ]:
import sys, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if sys.version_info[:2] not in {(3,12), (3,13)}:
    raise RuntimeError("Notebook dual-GPU này cần Python 3.12 hoặc 3.13; chọn Kaggle environment tương ứng.")
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Chọn accelerator có 2 GPU (ưu tiên T4 x2) trong Kaggle Settings.")

## 02. Nạp source MOESCORE

Gắn bản source chứa MOE-09.5 hoặc clone commit đã đẩy. Notebook dừng nếu source chưa chứa script hai GPU.


In [ ]:
import os, shutil, subprocess, json
for path in (WORK_ROOT, CONFIG_ROOT, LOG_ROOT, RUNS_ROOT): path.mkdir(parents=True, exist_ok=True)
if not PROJECT_ROOT.exists():
    if REPOSITORY_SOURCE is not None:
        shutil.copytree(REPOSITORY_SOURCE, PROJECT_ROOT, symlinks=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT_ROOT)], check=True)
required = ["scripts/kaggle_full9_dual_gpu.py", "scripts/kaggle_full9_artifacts.py",
            "scripts/extract_moe_feature_store.py", "scripts/train_moe_gating.py",
            "requirements-kaggle-py312.txt", "requirements-kaggle-py313.txt"]
missing = [name for name in required if not (PROJECT_ROOT/name).is_file()]
if missing: raise FileNotFoundError(f"Source chưa chứa MOE-09.5: {missing}; hãy gắn đúng source hoặc đẩy commit.")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Source:", PROJECT_ROOT)

## 03. Phụ thuộc theo phiên bản Python

Giữ bộ PyTorch/CUDA có sẵn. Python 3.12 dùng NumPy 1.26.4 và cài MS-CLAP/LAION-CLAP theo metadata gốc. Python 3.13 giữ chế độ NumPy 2 có kiểm tra smoke riêng; nhánh này vẫn có rủi ro tương thích. Nếu cài đặt đổi package đang được kernel nạp, restart kernel rồi chạy từ stage 00.


In [ ]:
import importlib, importlib.metadata as metadata
from scripts.kaggle_full9_dual_gpu import dependency_plan_for_python
requirements_file, legacy_no_deps = dependency_plan_for_python(sys.version_info)
print("Dependency plan:", requirements_file, "legacy metadata bypass:", legacy_no_deps)
loaded_before = {name: metadata.version(name) for name in ("numpy", "scipy", "transformers", "librosa") if name in sys.modules}
if INSTALL_MISSING_DEPENDENCIES:
    command = [sys.executable, str(PROJECT_ROOT/"scripts/install_kaggle_dependencies.py"),
               "--requirements", str(requirements_file),
               "--constraints", str(WORK_ROOT/"torch_constraints.txt"),
               "--report-json", str(LOG_ROOT/"dependencies_install.json")]
    if WHEELHOUSE: command += ["--wheelhouse", str(WHEELHOUSE)]
    subprocess.run(command, check=True)
    if legacy_no_deps:
        legacy = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if WHEELHOUSE: legacy += ["--no-index", "--find-links", str(WHEELHOUSE)]
        subprocess.run(legacy + ["msclap==1.3.4", "laion-clap==1.1.7"], check=True)
    importlib.invalidate_caches()
    changed_loaded = [name for name, old in loaded_before.items() if metadata.version(name) != old]
    if changed_loaded: raise RuntimeError("Các module đang nạp đã đổi phiên bản: " + ", ".join(changed_loaded) + ". Restart kernel rồi chạy lại từ stage 00.")
modules = {"torch":"torch", "torchaudio":"torchaudio", "torchvision":"torchvision",
           "transformers":"transformers", "huggingface_hub":"huggingface-hub",
           "laion_clap":"laion-clap", "msclap":"msclap", "timm":"timm",
           "librosa":"librosa", "torchlibrosa":"torchlibrosa", "nnAudio":"nnAudio",
           "numba":"numba", "soundfile":"soundfile", "scipy":"scipy",
           "einops":"einops", "kagglehub":"kagglehub", "kaggle":"kaggle"}
versions = {}
for module, distribution in modules.items():
    try:
        importlib.import_module(module)
        versions[distribution] = metadata.version(distribution)
    except Exception as exc:
        raise RuntimeError(f"Python {sys.version_info.major}.{sys.version_info.minor} dependency {distribution} import failed: {exc}; cung cấp wheel tương thích và chạy lại kernel.") from exc
if versions["msclap"] != "1.3.4": raise RuntimeError("Yêu cầu msclap==1.3.4")
(LOG_ROOT/"dependency_versions.json").write_text(json.dumps(versions, indent=2))
print(versions)

## 04. Khám phá Kaggle Inputs

Kiểm tra dataset, trọng số, HF cache và UniLM. Không suy diễn tên mount checkpoint.


In [ ]:
from scripts import kaggle_moe_pipeline as pipeline
print("Kaggle inputs:", [str(p) for p in sorted(Path("/kaggle/input").iterdir())])
if not DATASET_ROOT.exists():
    candidates = sorted({p.parent.parent for p in Path("/kaggle/input").rglob("meta_data/train_average.csv")})
    if len(candidates) != 1: raise FileNotFoundError(f"Chọn DATASET_ROOT chính xác; candidates={candidates}")
    DATASET_ROOT = candidates[0]
for split in ("train", "validation", "test"):
    for path in (DATASET_ROOT/f"meta_data/{split}_average.csv", DATASET_ROOT/f"wav/{split}"):
        if not path.exists(): raise FileNotFoundError(path)
if not WEIGHTS_ROOT.is_dir(): raise FileNotFoundError(f"Chỉnh WEIGHTS_ROOT: {WEIGHTS_ROOT}")
if not HF_CACHE_SOURCE.exists(): raise FileNotFoundError(f"Chỉnh HF_CACHE_SOURCE: {HF_CACHE_SOURCE}")
print("Dataset:", DATASET_ROOT, "Weights:", WEIGHTS_ROOT, "HF source:", HF_CACHE_SOURCE)

## 05. Khôi phục Hugging Face cache

Giữ nguyên cây Hub `blobs/refs/snapshots` và symlink tương đối. Dùng cache đã gắn hoặc tar archive, không tải model khi trích xuất.


In [ ]:
import tarfile
hub = HF_HOME/"hub"
if not hub.is_dir():
    if HF_CACHE_SOURCE.is_dir():
        candidate = HF_CACHE_SOURCE/"hub" if (HF_CACHE_SOURCE/"hub").is_dir() else HF_CACHE_SOURCE
        if not any(candidate.glob("models--*")): raise ValueError("HF cache source thiếu models--* trong hub")
        shutil.copytree(candidate, hub, symlinks=True)
    else:
        HF_HOME.mkdir(parents=True, exist_ok=True)
        with tarfile.open(HF_CACHE_SOURCE) as archive:
            archive.extractall(HF_HOME, filter="data")
        if not hub.is_dir(): raise ValueError("HF archive phải chứa thư mục hub/ nguyên cấu trúc")
for name in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE"):
    os.environ[name] = str(HF_HOME if name == "HF_HOME" else hub)
for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ[name] = "1"
print("Hub cache:", hub, "models:", [p.name for p in hub.glob("models--*")])

## 06. Tạo cấu hình full_9

Tạo bản YAML trong /kaggle/working. Tùy chọn `CHECKPOINT_OVERRIDES` dùng tên expert.field; không sửa source.


In [ ]:
from scripts.extract_moe_feature_store import FULL9_CHECKPOINT_FIELDS
from model.expert_adapters.registry import EXPERT_PROFILES
from scripts.kaggle_full9_dual_gpu import validate_worker_devices
from scripts import kaggle_full9_artifacts as artifacts
import yaml
order = EXPERT_PROFILES["full_9"]
validate_worker_devices(WORKER_DEVICES, order)
defaults = {
 "expert1":{"pretrained_checkpoint":"pretrained/laion-clap.pt","finetuned_checkpoint":"finetuned_weights/expert1_best.pt"},
 "expert2":{"pretrained_checkpoint":"pretrained/mga-clap.pt","finetuned_checkpoint":"finetuned_weights/expert2_best.pt"},
 "expert3":{"base_checkpoint":"pretrained/checkpoint-30.pth","finetuned_checkpoint":"finetuned_weights/expert3_best.pt"},
 "expert4":{"beats_checkpoint":"pretrained/BEATs_iter3_plus_AS2M.pt","clap_checkpoint":"pretrained/laion-clap.pt","finetuned_checkpoint":"finetuned_weights/expert4_best.pt"},
 "top2_m2d":{"checkpoint":"pretrained/checkpoint-30.pth"},
 "top2_msclap":{"checkpoint":"pretrained/CLAP_weights_2023.pth"},
 "top2_laion":{"checkpoint":"pretrained/laion-clap.pt"},
 "top2_mga":{"pretrained_checkpoint":"pretrained/mga-clap.pt"},
 "m2d_lora":{"base_checkpoint":"pretrained/checkpoint-30.pth","finetuned_checkpoint":"best.pt"}}
for key, path in CHECKPOINT_OVERRIDES.items():
    expert, field = key.split(".", 1)
    if expert not in defaults or field not in defaults[expert]: raise ValueError(f"Unknown checkpoint override: {key}")
    defaults[expert][field] = str(path)
expert_config, gate_config, resolved = pipeline.generate_configs(
    PROJECT_ROOT, DATASET_ROOT, WEIGHTS_ROOT, HF_HOME, FEATURE_CACHE_ROOT, RUNS_ROOT,
    CONFIG_ROOT, expert_checkpoints=defaults, profile="full_9", unilm_source_dir=UNILM_SOURCE_DIR)
config = yaml.safe_load(gate_config.read_text())
config["runtime"].update(device=GATE_DEVICE, seed=SEED)
config["training"].update(batch_size=TRAIN_BATCH_SIZE, max_epochs=TRAIN_EPOCHS, learning_rate=TRAIN_LEARNING_RATE)
gate_config.write_text(yaml.safe_dump(config, sort_keys=False))
(CONFIG_ROOT/"worker_devices.json").write_text(json.dumps(WORKER_DEVICES, indent=2))
print("Expert config:", expert_config, "Gate config:", gate_config)

## 07. Offline preflight

Kiểm chứng đủ 9 checkpoint, cấu trúc dataset, source UniLM, import dependency và snapshot text model trước khi nạp expert.


In [ ]:
from configs.moe_config import load_moe_config, resolve_moe_paths
preflight_report = pipeline.preflight(resolve_moe_paths(load_moe_config(expert_config), expert_config), DATASET_ROOT,
                                      WORK_ROOT/"preflight", require_cuda=True, check_cache=True)
if preflight_report["status"] != "PASS": raise RuntimeError(preflight_report)
subprocess.run([sys.executable, str(PROJECT_ROOT/"scripts/prepare_moe_text_cache.py"),
                "--cache-dir", str(HF_HOME), "--profile", "full_9"], check=True)
print("Preflight PASS:", preflight_report["splits"])

## 08. Smoke trích xuất thực trên hai GPU

4 mẫu mỗi split, cả 9 expert; mỗi GPU chỉ giữ một model. Log từng expert trong `smoke/logs/`. Dừng nếu có worker lỗi.


In [ ]:
def run_cli(*parts):
    command = [sys.executable, *map(str, parts)]
    print("RUN", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
if RUN_GPU_SMOKE:
    smoke_report = LOG_ROOT/"smoke_extraction.json"
    run_cli(PROJECT_ROOT/"scripts/kaggle_full9_dual_gpu.py", "--config", expert_config,
            "--dataset-root", DATASET_ROOT, "--output-root", SMOKE_CACHE_ROOT,
            "--worker-devices-json", CONFIG_ROOT/"worker_devices.json", "--smoke",
            "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[n]) for n in order),
            "--shard-size", SHARD_SIZE, "--report-json", smoke_report)
    smoke_result = json.loads(smoke_report.read_text())
    if smoke_result["status"] != "PASS": raise RuntimeError(smoke_result)
    artifacts.validate_full9_store(smoke_result["feature_store_root"], {s:4 for s in ("train","validation","test")})
    print("Smoke GPU assignment:", smoke_result["worker_devices"])
    for name, info in smoke_result.get("workers", {}).items():
        print(name, info["device"], info["gpu_name"], "peak allocated MiB",
              round(info["peak_allocated_vram_bytes"] / 2**20, 1), "loaded", info["model_constructed"])
else:
    raise RuntimeError("Full execution requires RUN_GPU_SMOKE=True or an explicitly verified prior smoke stage")

## 09. Smoke gate trên đặc trưng thật

Ba bước optimizer với feature store vừa trích xuất; không nạp chín expert cùng lúc.


In [ ]:
if RUN_GPU_SMOKE:
    smoke_gate = pipeline.smoke_store(smoke_result["feature_store_root"], config)
    if smoke_gate["status"] != "PASS" or smoke_gate["optimizer_steps"] != 3: raise RuntimeError(smoke_gate)
    (LOG_ROOT/"smoke_gate.json").write_text(json.dumps(smoke_gate, indent=2))
    print("Gate smoke:", smoke_gate)

## 10. Trích xuất full_9 hai GPU

Chỉ chạy khi `SMOKE_ONLY=False` và `RUN_FULL_EXTRACTION=True`. Job hash checkpoint/sample một lần, hai queue GPU chạy song song, sau đó assemble.


In [ ]:
if not SMOKE_ONLY and RUN_FULL_EXTRACTION:
    final_report = LOG_ROOT/"full_extraction.json"
    run_cli(PROJECT_ROOT/"scripts/kaggle_full9_dual_gpu.py", "--config", expert_config,
            "--dataset-root", DATASET_ROOT, "--output-root", FEATURE_CACHE_ROOT,
            "--worker-devices-json", CONFIG_ROOT/"worker_devices.json",
            "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[n]) for n in order),
            "--shard-size", SHARD_SIZE, "--report-json", final_report)
    full_result = json.loads(final_report.read_text())
    if full_result["status"] != "PASS": raise RuntimeError(full_result)
elif SMOKE_ONLY:
    print("SMOKE_ONLY: bỏ qua trích xuất đầy đủ")

## 11. Xác thực feature store

Kiểm tra SHA-256, profile, thứ tự, số dòng 7500/3000/3000 và width 27399. Cache cũ chỉ được dùng lại khi job provenance trùng.


In [ ]:
if not SMOKE_ONLY:
    if not RUN_FULL_EXTRACTION and not EXISTING_FEATURE_CACHE:
        raise ValueError("Bật RUN_FULL_EXTRACTION hoặc cung cấp EXISTING_FEATURE_CACHE đã xác thực")
    if EXISTING_FEATURE_CACHE and not RUN_FULL_EXTRACTION:
        mounted = Path(EXISTING_FEATURE_CACHE)
        if not (FEATURE_CACHE_ROOT/"feature_cache").exists():
            if (mounted/"upload_manifest.json").is_file(): artifacts.restore_feature_cache(mounted, FEATURE_CACHE_ROOT)
            elif (mounted/"feature_cache").is_dir(): shutil.copytree(mounted/"feature_cache", FEATURE_CACHE_ROOT/"feature_cache", symlinks=True)
            else: raise FileNotFoundError("Existing cache thiếu archive hoặc feature_cache/")
        # Chỉ tin cache khi source/checkpoints và identity hiện tại tái lập đúng.
        from scripts.extract_moe_feature_store import prepare_job, completed_stores_match, assembly_store_root
        _, expected_job, expected_manifests = prepare_job(expert_config, DATASET_ROOT, FEATURE_CACHE_ROOT,
            batch_sizes=EXTRACTION_BATCH_SIZES, shard_size=SHARD_SIZE, profile="full_9", worker_devices=WORKER_DEVICES)
        if not completed_stores_match(expected_job, expected_manifests, FEATURE_CACHE_ROOT):
            raise ValueError("Existing feature cache incomplete; bật RUN_FULL_EXTRACTION để resume")
        full_result = {"status":"PASS", "feature_store_root":str(assembly_store_root(FEATURE_CACHE_ROOT, expected_job)),
                       "assembly_id":__import__("scripts.extract_moe_feature_store", fromlist=["job_identity"]).job_identity(expected_job)}
    full_store = Path(full_result["feature_store_root"])
    store_validation = artifacts.validate_full9_store(full_store, {"train":7500,"validation":3000,"test":3000})
    config["data"]["feature_store_root"] = str(full_store)
    gate_config.write_text(yaml.safe_dump(config, sort_keys=False))
    (LOG_ROOT/"full_store_validation.json").write_text(json.dumps(store_validation, indent=2))
    print(store_validation)

## 12. Upload cache lên Kaggle Dataset

Chỉ upload `feature_cache.tar` và manifest SHA. `kagglehub.dataset_upload` tạo dataset hoặc version; sau upload tải lại manifest để kiểm chứng. Dừng trước training nếu upload lỗi. Dataset mới giữ private mặc định.


In [ ]:
if not SMOKE_ONLY:
    if UPLOAD_FEATURE_CACHE:
        package = artifacts.package_feature_cache(FEATURE_CACHE_ROOT, UPLOAD_DIRECTORY, stores=store_validation)
        upload_report = artifacts.upload_feature_cache(FEATURE_DATASET_HANDLE, UPLOAD_DIRECTORY,
            version_notes=f"MOESCORE full_9 {full_result['assembly_id']}")
        (LOG_ROOT/"upload.json").write_text(json.dumps(upload_report, indent=2))
        print("Uploaded:", upload_report["url"])
    elif RUN_GATE_TRAINING:
        print("UPLOAD_FEATURE_CACHE=False được chọn rõ ràng; tiếp tục training trên cache cục bộ")

## 13. Train Gating Network

Chỉ học gate 27399→512→9. Dùng feature store, không nạp expert. Resume từ `latest_gate.pt` nếu đã có.


In [ ]:
if not SMOKE_ONLY and RUN_GATE_TRAINING:
    latest = RUNS_ROOT/"latest_gate.pt"
    args = [PROJECT_ROOT/"scripts/train_moe_gating.py", "--config", gate_config,
            "--device", GATE_DEVICE, "--report-json", LOG_ROOT/"training.json"]
    if latest.is_file(): args += ["--resume", latest]
    run_cli(*args)
    train_report = json.loads((LOG_ROOT/"training.json").read_text())
    if train_report["status"] != "PASS": raise RuntimeError(train_report)
    print("Best:", train_report.get("best_checkpoint"), "Latest:", train_report.get("latest_checkpoint"))

## 14. Đánh giá validation

Chỉ dùng best checkpoint do validation SRCC chọn. Không dùng test labels.


In [ ]:
if not SMOKE_ONLY and RUN_EVALUATION:
    best = RUNS_ROOT/"best_gate.pt"
    if not best.is_file(): raise FileNotFoundError("best_gate.pt chưa có; kiểm tra SRCC validation/training")
    run_cli(PROJECT_ROOT/"scripts/evaluate_moe_gating.py", "--config", gate_config,
            "--checkpoint", best, "--device", GATE_DEVICE,
            "--report-json", RUNS_ROOT/"validation_metrics.json")
    from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features
    from data.moe_feature_store import preprocessing_contract
    from model.moe_gating_core import GatingNetwork
    from training.checkpoint_manager import MoeCheckpointManager
    from training.moe_evaluator import MoeGateEvaluator, write_submission_csv
    from torch.utils.data import DataLoader
    validation = MoeTrainingDataset(str(full_store), split="validation", require_labels=True)
    gate = GatingNetwork(input_feature_dim=validation.manifest["feature_width"], num_experts=9, profile="full_9").to(GATE_DEVICE)
    payload = MoeCheckpointManager().load_for_inference(best, gating_network=gate,
        expected_expert_checkpoint_provenance=validation.manifest["checkpoint_provenance"])
    if payload["feature_store_identity"]["validation"] != validation.identity: raise ValueError("Validation identity mismatch")
    if payload["feature_store_identity"]["preprocessing_contract"] != preprocessing_contract(validation.manifest): raise ValueError("Preprocessing identity mismatch")
    batches = DataLoader(validation, batch_size=config["validation"]["batch_size"], shuffle=False,
                         collate_fn=collate_moe_features)
    results = MoeGateEvaluator(gate).evaluate(batches)
    rows = [{"sample_id":sid,"wave_file_name":wav,"pred_score":float(score)} for sid,wav,score in
            zip(results["sample_ids"], results["wave_file_names"], results["prediction_raw"].tolist())]
    write_submission_csv(rows, RUNS_ROOT/"validation_predictions.csv")
    print(json.loads((RUNS_ROOT/"validation_metrics.json").read_text()))

## 15. Dự đoán test không nhãn

Tạo `submission.csv` theo thứ tự gốc, không đọc nhãn test. Lưu thêm `test_predictions.csv`.


In [ ]:
if not SMOKE_ONLY and RUN_TEST_INFERENCE:
    best = RUNS_ROOT/"best_gate.pt"
    if not best.is_file(): raise FileNotFoundError(best)
    run_cli(PROJECT_ROOT/"scripts/infer_moe_gating.py", "--config", gate_config,
            "--checkpoint", best, "--device", GATE_DEVICE,
            "--output", RUNS_ROOT/"submission.csv", "--report-json", RUNS_ROOT/"test_inference.json")
    shutil.copy2(RUNS_ROOT/"submission.csv", RUNS_ROOT/"test_predictions.csv")
    print("Submission:", RUNS_ROOT/"submission.csv")

## 16. Artifacts và đường dẫn

Đối chiếu log, checkpoint, dự đoán và dataset URL. Smoke và full cache được lưu riêng.


In [ ]:
artifacts_to_show = [LOG_ROOT/"smoke_extraction.json", LOG_ROOT/"smoke_gate.json",
                     LOG_ROOT/"full_extraction.json", LOG_ROOT/"full_store_validation.json",
                     LOG_ROOT/"upload.json", RUNS_ROOT/"best_gate.pt",
                     RUNS_ROOT/"validation_metrics.json", RUNS_ROOT/"submission.csv"]
for path in artifacts_to_show:
    print("PRESENT" if path.exists() else "SKIPPED", path)
print("Feature dataset:", f"https://www.kaggle.com/datasets/{FEATURE_DATASET_HANDLE}" if (LOG_ROOT/"upload.json").is_file() else "chưa upload")